# Process CSDA Evaluation Sites AOIs to a `GeoJSON` polygon

Paul Montesano, PhD  
April 2026

In [1]:
import pandas as pd
import geopandas as gpd
from urllib.parse import quote
import numpy as np

import sys

sys.path.append('/projects/code/csda_summaries/lib')
sys.path.append('/home/pmontesa/code/csda_summaries/lib')

import siteslib

from datetime import datetime

## Evaluation Sites

#### Set default site AOI extent size (either a radius, or a box side - depending on `aoi_type` field)

In [2]:
BUF_KM = 3

In [3]:
# Define your spreadsheet ID and sheet name
SPREADSHEET_ID = '13MrpqFtAOqQY9WdW9lHNsqjCbG-e3VQkEDbHOGIKa6k'

#### Read in sites database

In [4]:
SHEET_NAME = 'Evaluation Sites' # The name of the specific tab

# 2. Encode the sheet name for safe use in a URL
ENCODED_SHEET_NAME = quote(SHEET_NAME)

# 3. Construct the full URL using the gviz/tq endpoint
url = f"https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/gviz/tq?tqx=out:csv&sheet={ENCODED_SHEET_NAME}"

# 4. Use pandas to read the CSV data directly from the URL
try:
    sites = pd.read_csv(url)
    sites['Site Name'] = sites['Site Name abbrev'].str.rstrip()
    #sites = sites[sites['Site Name'] != 'Sicily'] # Sicily site is same as Catania
    
except Exception as e:
    print(f"An error occurred: {e}")

# Get only Priority Sites - used for defining CSDA sites for now
sites = sites[sites['Priority Level'] != 'unknown']

# # Get only Priority Sites = high
# sites = sites[sites['Priority Level'] == 'high']

# Get the list of columns to drop
cols_to_drop = sites.columns[sites.columns.str.contains('Unnamed')]
sites = sites.drop(columns=cols_to_drop)

## Create site AOIs

#### Add all custom site AOI geojsons here

In [5]:
custom_geojson_dict = {
    'PICS Libya-4': '/explore/nobackup/projects/CSDA_eval/sites/map_1x3_MODISgrids_Libya4.geojson'
}

In [23]:
# Create GeoDataFrame
sites_gdf = siteslib.create_sites_gdf_with_aois(
    sites, 
    default_size_km=BUF_KM,
    custom_geojson_dict=custom_geojson_dict, 
    site_column='Site Name',
    lat_column='Latitude',
    lon_column='Longitude'
)

# Calc area
#sites_gdf['aoi_area_km2'] = sites_gdf.to_crs(sites_gdf.estimate_utm_crs()).area / 1e6

print(f"\n{sites_gdf.shape[0]} sites are being used to search for imagery.")

✓ Loaded custom geometry for PICS Libya-4

115 sites are being used to search for imagery.


In [24]:
def calculate_area_km2_per_site(gdf):
    """
    Calculate area in km² for each feature in its own UTM zone
    (More accurate for global datasets)
    """
    areas = []
    
    for idx, row in gdf.iterrows():
        # Create single-feature GeoDataFrame
        single_gdf = gpd.GeoDataFrame([row], geometry='geometry', crs=gdf.crs)
        
        # Project to appropriate UTM zone for this feature
        single_proj = single_gdf.to_crs(single_gdf.estimate_utm_crs())
        
        # Calculate area in km²
        area_km2 = single_proj.area.iloc[0] / 1_000_000
        areas.append(area_km2)
    
    return areas

# Usage
sites_gdf['area_km2'] = calculate_area_km2_per_site(sites_gdf)

#### Remove misc fields

In [25]:
rm_cols_list = ['Site min sqkm', 'Max View Angle',
       'Min # of Acqs/Sensor', 'Max Cloud %', 'acquisition sqkm', 'Notes',
       'UL (Latitude, Longitude)', 'LL (Latitude, Longitude)',
       'UR (Latitude, Longitude)', 'LR (Latitude, Longitude)']

sites_gdf.drop(rm_cols_list, axis=1, inplace=True)

#### Save sites `GeoJSON` and copy to `GitHub` repo

In [26]:
csda_sites_fn = f'/explore/nobackup/projects/CSDA_eval/sites/csda_sites_aoi.geojson'
csda_sites_fn

'/explore/nobackup/projects/CSDA_eval/sites/csda_sites_aoi.geojson'

In [27]:
sites_gdf.to_file(csda_sites_fn)

In [28]:
!eval cp $csda_sites_fn /home/pmontesa/code/csda_summaries/sites/

In [29]:
sites_gdf

,Site Name abbrev,Site Name,Location Name,Country,Longitude,Latitude,Remote Sensing Domain,Priority Level,Evaluation Category,Source,Surface Domain,Assessment type(s),aoi_type,aoi_size_km,geometry,area_km2
0,Albuquerque,Albuquerque,New Mexico,USA,-106.613826,35.068706,Optical Multi,high,Geometric,NASA,Land,Geometric Calibration Quality (priority pointi...,box,3.00,"POLYGON ((-106.59712 35.05540, -106.59764 35.0...",9.000000
2,Baotou,Baotou,China cal/val,China,109.629437,40.851787,Optical Multi/Hyper,high,Radiometric & Geometric,RadCalNet,Land,"Radiometric, Geometric Calibration Quality (LS...",box,0.04,"POLYGON ((109.62968 40.85161, 109.62967 40.851...",0.001600
3,Belo Horizonte,Belo Horizonte,Belo Horizonte,Brazil,-43.943230,-19.892135,Optical Multi,high,Geometric,NASA,Land,Geometric Calibration Quality (additional poin...,box,3.00,"POLYGON ((-43.92881 -19.90560, -43.92899 -19.8...",9.000000
4,Boston,Boston,Massachusetts,USA,-71.009400,42.367454,Optical Multi,high,Geometric,unknown,Land,Geometric Calibration Quality (priority pointi...,box,3.00,"POLYGON ((-70.99077 42.35427, -70.99162 42.381...",9.000000
6,Cape Town,Cape Town,Cape Town,South Africa,18.507113,-33.992986,Optical Multi/Hyper,high,Geometric,unknown,Land,Geometric Calibration Quality (additional poin...,box,3.00,"POLYGON ((18.52295 -34.00683, 18.52373 -33.979...",9.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,UT FEMA FS Flaming Gorge 3,UT FEMA FS Flaming Gorge 3,UT_FEMA_FS_FlamingGorge_2020_B20_3,USA,-111.530000,41.405370,DEM,medium,Geometric,USGS,Land,DEM Horizontal and Vertical Accuracy,circle,NaN,"POLYGON ((-111.49411 41.40553, -111.49426 41.4...",28.228936
114,UT FEMA FS Flaming Gorge 4,UT FEMA FS Flaming Gorge 4,UT_FEMA_FS_FlamingGorge_2020_B20_4,USA,-111.173000,39.646440,DEM,medium,Geometric,USGS,Land,DEM Horizontal and Vertical Accuracy,circle,NaN,"POLYGON ((-111.13803 39.64649, -111.13820 39.6...",28.228936
115,UT West East,UT West East,UT_WestEast_B22_1,USA,-110.387000,40.656560,DEM,medium,Geometric,USGS,Land,DEM Horizontal and Vertical Accuracy,circle,NaN,"POLYGON ((-110.35151 40.65637, -110.35171 40.6...",28.228936
116,WA North Central,WA North Central,WA_NorthCentral_2021_B21_1,USA,-120.646000,48.786160,DEM,medium,Geometric,USGS,Land,DEM Horizontal and Vertical Accuracy,circle,NaN,"POLYGON ((-120.60519 48.78532, -120.60552 48.7...",28.228936


In [30]:
sites_gdf[sites_gdf['Site Name'] == 'PICS Libya-4']

,Site Name abbrev,Site Name,Location Name,Country,Longitude,Latitude,Remote Sensing Domain,Priority Level,Evaluation Category,Source,Surface Domain,Assessment type(s),aoi_type,aoi_size_km,geometry,area_km2
28,PICS Libya-4,PICS Libya-4,PICS Libya-4,Libya,23.39,28.55,Optical Multi,high,Radiometric,USGS,Land,Radiometric Calibration Quality (Temporal),custom,NaN,"POLYGON ((23.39665 28.55833, 23.39110 28.53333...",2.573687
